In [1]:
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt
from numpy.linalg import inv
import os
from os import listdir
from os.path import isfile, isdir, join
from os import walk

from colormath.color_conversions import convert_color
from colormath.color_objects import LabColor, XYZColor
from colormath.color_diff import delta_e_cie2000

import skimage
from skimage.color import rgb2lab, lab2rgb

from scipy.interpolate import BSpline, make_interp_spline
from scipy.optimize import minimize
from scipy.optimize import dual_annealing
from scipy.optimize import least_squares
from scipy.optimize import basinhopping

from numpy.random import random
from numpy.random import rand

from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

import time

In [2]:
table1 = np.load('cap1.npy')
table2 = np.load('cap3_2.npy')
table3 = np.load('cap5.npy')

In [3]:
def transfer_by_table(img, table):
    
    # reshape as nx3, and to int
    temp_img = img.reshape(-1, 3).astype(int)

    # transfer to table key
    img_keys = temp_img[:,0] + np.left_shift(temp_img[:,1], 8) + np.left_shift(temp_img[:,2], 16)

    # table mapping
    img_values = table[img_keys]

    # reconstruct to rgb image
    img_r = img_values.astype('uint8')
    img_g = np.right_shift(img_values, 8).astype('uint8')
    img_b = np.right_shift(img_values, 16).astype('uint8')

    img_trans = np.column_stack((img_r, img_g, img_b))

    img_trans = img_trans.reshape(img.shape)
    
    return img_trans

In [4]:
def filter_ext(files, tar_ext):
    
    tar_files = []
    
    for file in files:
        
        file_ext = os.path.splitext(file)[-1][1:].lower()
        
        if file_ext == tar_ext:
            tar_files.append(file)
        
    return tar_files
    
def get_all_ext_list(tar_path, tar_ext):
    
    all_tar_files = []
    
    for root, dirs, files in walk(tar_path):
        
        tar_files = filter_ext(files, tar_ext.lower())

        if len(tar_files) == 0:
            continue
            
        tar_files = [join(root, file) for file in tar_files]
            
        all_tar_files = all_tar_files + tar_files
        
    return all_tar_files

In [5]:
def to_nbi(wli, is_dual = False):
    h,w,c = wli.shape

    image_rgb = cv2.cvtColor(wli, cv2.COLOR_BGR2RGB)

    pred_img2 = transfer_by_table(image_rgb, table2)

    pred_img2 = cv2.cvtColor(pred_img2, cv2.COLOR_RGB2BGR)
    
    if is_dual:
        tar_img = np.zeros((h, w*2, c))
        tar_img[:h,:w,:] = wli
        tar_img[:h,w:2*w,:] = pred_img2
    else:
        tar_img = np.zeros((h, w, c))
        tar_img[:h,:w,:] = pred_img2
        
    tar_img = tar_img.astype('uint8')
    
    return tar_img

In [6]:
src_path = r'F:\WLI\test\Angiectasia'
tar_path = r'F:\SAVE\test\Angiectasia'

In [7]:
res_files = get_all_ext_list(src_path, 'jpg')

In [8]:
is_dual = False

for index, file in enumerate(res_files):
    
    dump_file = file.replace(src_path, tar_path)
    dump_file_path = os.path.dirname(os.path.abspath(dump_file))
    
    if not os.path.isdir(dump_file_path):
        try:
            os.makedirs(dump_file_path)
        except FileExistsError:
            print("建立路徑異常。")
            
    img = cv2.imread(file)
#     img = img[22:922, 694:1894,:]
    nbi = to_nbi(img, is_dual=is_dual)
    
    cv2.imwrite(dump_file, nbi)
    
    print(index, dump_file)

0 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_11607.jpg
1 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_11608.jpg
2 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_11611.jpg
3 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_5619.jpg
4 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_5624.jpg
5 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_5641.jpg
6 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_7670.jpg
7 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_890.jpg
8 F:\SAVE\test\Angiectasia\04a78ef00c5245e0_895.jpg
9 F:\SAVE\test\Angiectasia\64440803f87b4843_18235.jpg
10 F:\SAVE\test\Angiectasia\64440803f87b4843_18329.jpg
11 F:\SAVE\test\Angiectasia\64440803f87b4843_18381.jpg
12 F:\SAVE\test\Angiectasia\64440803f87b4843_18384.jpg
13 F:\SAVE\test\Angiectasia\64440803f87b4843_18385.jpg
14 F:\SAVE\test\Angiectasia\64440803f87b4843_18458.jpg
15 F:\SAVE\test\Angiectasia\64440803f87b4843_18461.jpg
16 F:\SAVE\test\Angiectasia\64440803f87b4843_18514.jpg
17 F:\SAVE\test\Angiectasia\64440803f87b4843_18526.jpg
18 F:\SAVE\test\Angiectasia\